In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
db_path = r'C:\Users\Admin\Desktop\DA Project\E-Commerce Sales Analytics\E-Commerce Sales.db'
conn = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM master_customer_summary", conn)
conn.close()

In [3]:
df = df.dropna(subset=['recency_days', 'total_orders', 'total_net_sales', 'cum_profit_pct'])

In [4]:
df['R_score'] = pd.qcut(df['recency_days'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1]).astype(int)
df['F_score'] = pd.qcut(df['total_orders'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
df['M_score'] = pd.qcut(df['total_net_sales'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

In [5]:
r, f, m = df['R_score'], df['F_score'], df['M_score']
df['rfm_segment'] = np.select(
    [(r >= 4) & (f >= 4) & (m >= 4), (r >= 3) & (f >= 3), (r >= 3) & (f <= 2), (r <= 2) & (f >= 3), ],
    ['Champions', 'Loyal Customers', 'Potential Loyalists', 'At Risk'], default='Lost / Hibernating')

In [6]:
df = df.sort_values('cum_profit_pct')
prev_cum = df['cum_profit_pct'].shift(fill_value=0)
df['is_pareto_80'] = np.where(prev_cum < 80, 'Top 80% Profit (Pareto)', 'Remaining 20% Profit')

In [7]:
output_path = r'C:\Users\Admin\Desktop\DA Project\E-Commerce Sales Analytics\final_customer_analytics.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')

In [8]:
print(df['rfm_segment'].value_counts())

rfm_segment
Loyal Customers        6954
Lost / Hibernating     5560
Potential Loyalists    4405
At Risk                4404
Champions              3588
Name: count, dtype: int64
